In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression


In [2]:
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


os.chdir("/notebooks/Thesis-DataLogging")
print("Changed working directory to:", os.getcwd())

Current working directory: /notebooks/Thesis-DataLogging
Files in this directory: ['CSV_SQLite.ipynb', '.ipynb_checkpoints', 'FlowLog_20251127_113337.csv', 'flow_data.db', '40min_print_SI383820251201085118_PartStatistics_all.csv', 'Airbearing_print_271125_SI383820251128093848_PartStatistics_all.csv', 'Merged_Data_log_OT.csv', 'Timestamp_LA_INDEX_27.csv', 'Timestamp_LA_INDEX.csv', 'Merged_Data_log_OT_27.csv', '27_Merged_Data_log_OT.csv', 'FlowLog_20251203_123354.csv', 'FlowLog_20251203_145757.csv', '80um-reference-job-SI383820251203121902_PartStatistics_all.csv', 'merged_3_12_25.csv', 'Timestamp_LA_INDEX_03.csv', '03_Merged_Data_log_OT.csv', 'SI383820251203121902 (2).pdf', '03_12_25_Merged_Data_log_OT.csv', 'extracted_data.csv', 'Fullmerged.csv', 'FlowLog_20251215_095220.csv', 'DoE.ipynb', 'Run_2.pdf', 'Run_1.pdf', 'Sensor_Calibration(Sheet1).csv', 'ML.ipynb', 'plc_operational_data_with_corrected_flows.csv', 'Backend.ipynb']
Changed working directory to: /notebooks/Thesis-DataLogging


In [3]:
# Current working directory should be .../notebooks
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR

INPUT_PLC_CSV = DATA_DIR / "FlowLog_20251215_095220.csv"
INPUT_CALIB_CSV = DATA_DIR / "Sensor_Calibration(Sheet1).csv"
OUTPUT_CSV = DATA_DIR / "plc_operational_data_with_corrected_flows.csv"

# Safety checks
assert INPUT_PLC_CSV.exists(), f"Missing file: {INPUT_PLC_CSV}"
assert INPUT_CALIB_CSV.exists(), f"Missing file: {INPUT_CALIB_CSV}"

print("Paths resolved correctly")


Paths resolved correctly


In [4]:
def load_operational_data(path):
    df = pd.read_csv(path)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.sort_values("Timestamp").reset_index(drop=True)
    return df


def load_calibration_data(path):
    """
    Columns expected:
    SensorType, RAW_value, Flow_Lmin
    """
    return pd.read_csv(path)


plc_df = load_operational_data(INPUT_PLC_CSV)
calib_df = load_calibration_data(INPUT_CALIB_CSV)

print("Operational data shape:", plc_df.shape)
print("Calibration data shape:", calib_df.shape)


Operational data shape: (165587, 9)
Calibration data shape: (11, 3)


In [5]:
"""
def build_sensor_training_data(
    op_df,
    cal_df,
    sensor_name,
    raw_col,
    flow_col
):
    # Operational data
    df_op = op_df[[raw_col, flow_col]].copy()
    df_op["source"] = "operational"

    # Calibration anchors
    df_cal = cal_df[cal_df["SensorType"] == sensor_name].copy()
    df_cal = df_cal.rename(
        columns={"RAW_value": raw_col, "Flow_Lmin": flow_col}
    )
    df_cal["source"] = "calibration"

    return pd.concat([df_op, df_cal], ignore_index=True)
"""

'\ndef build_sensor_training_data(\n    op_df,\n    cal_df,\n    sensor_name,\n    raw_col,\n    flow_col\n):\n    # Operational data\n    df_op = op_df[[raw_col, flow_col]].copy()\n    df_op["source"] = "operational"\n\n    # Calibration anchors\n    df_cal = cal_df[cal_df["SensorType"] == sensor_name].copy()\n    df_cal = df_cal.rename(\n        columns={"RAW_value": raw_col, "Flow_Lmin": flow_col}\n    )\n    df_cal["source"] = "calibration"\n\n    return pd.concat([df_op, df_cal], ignore_index=True)\n'

In [6]:
"""
def train_hybrid_sensor_model(df, raw_col, flow_col):
    X = df[[raw_col, flow_col]]
    y = df[flow_col]

    # Give calibration points higher importance
    sample_weight = np.where(
        df["source"] == "calibration", 5.0, 1.0
    )

    model = LGBMRegressor(
        n_estimators=800,
        learning_rate=0.04,
        max_depth=5,
        monotone_constraints=[1, 1],
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42
    )

    tscv = TimeSeriesSplit(n_splits=5)
    maes = []

    for tr, te in tscv.split(X):
        model.fit(
            X.iloc[tr],
            y.iloc[tr],
            sample_weight=sample_weight[tr]
        )
        preds = model.predict(X.iloc[te])
        maes.append(mean_absolute_error(y.iloc[te], preds))

    print(f"MAE: {np.mean(maes):.4f} L/min")

    model.fit(X, y, sample_weight=sample_weight)
    return model
"""

'\ndef train_hybrid_sensor_model(df, raw_col, flow_col):\n    X = df[[raw_col, flow_col]]\n    y = df[flow_col]\n\n    # Give calibration points higher importance\n    sample_weight = np.where(\n        df["source"] == "calibration", 5.0, 1.0\n    )\n\n    model = LGBMRegressor(\n        n_estimators=800,\n        learning_rate=0.04,\n        max_depth=5,\n        monotone_constraints=[1, 1],\n        subsample=0.9,\n        colsample_bytree=0.9,\n        random_state=42\n    )\n\n    tscv = TimeSeriesSplit(n_splits=5)\n    maes = []\n\n    for tr, te in tscv.split(X):\n        model.fit(\n            X.iloc[tr],\n            y.iloc[tr],\n            sample_weight=sample_weight[tr]\n        )\n        preds = model.predict(X.iloc[te])\n        maes.append(mean_absolute_error(y.iloc[te], preds))\n\n    print(f"MAE: {np.mean(maes):.4f} L/min")\n\n    model.fit(X, y, sample_weight=sample_weight)\n    return model\n'

In [8]:
def build_sensor_training_data_pure(cal_df, sensor_name, raw_col):
    df_cal = cal_df[cal_df["SensorType"] == sensor_name].copy()
    df_cal = df_cal.rename(
        columns={"RAW_value": raw_col, "Flow_Lmin": "True_Lmin"}
    )
    return df_cal  # only calibration

def train_sensor_model(df, raw_col):
    X = df[[raw_col]]
    y = df["True_Lmin"]
    """
    model = LGBMRegressor(
        n_estimators=10,
        learning_rate=0.05,
        max_depth=1,
        num_leaves=2,
        min_data_in_leaf=1,
        subsample=1.0,
        colsample_bytree=1.0,
        random_state=42
    )
    """
    model=LinearRegression()
    model.fit(X, y)
    return model

def apply_all_sensor_models_fixed(op_df, cal_df):
    sensor_specs = [
        ("LowFlow", "LowFlowRAW", "LowFlow_Lmin"),
        ("ArgonFlow", "ArgonFlowRAW", "ArgonFlow_Lmin"),
        ("HighFlow", "HighFlowRAW", "HighFlow_Lmin"),
    ]

    for sensor, raw_col, out_col in sensor_specs:
        print(f"\nTraining model for {sensor} sensor")
        train_df = build_sensor_training_data_pure(cal_df, sensor, raw_col)
        model = train_sensor_model(train_df, raw_col)
        op_df[out_col] = model.predict(op_df[[raw_col]])

    return op_df

plc_df = apply_all_sensor_models_fixed(plc_df, calib_df)



Training model for LowFlow sensor

Training model for ArgonFlow sensor

Training model for HighFlow sensor


In [9]:
"""
def apply_all_sensor_models(op_df, cal_df):

    sensor_specs = [
        ("LowFlow", "LowFlowRAW", "LowFlow", "LowFlow_Lmin"),
        ("ArgonFlow", "ArgonFlowRAW", "ArgonFlow", "ArgonFlow_Lmin"),
        ("HighFlow", "HighFlowRAW", "HighFlow", "HighFlow_Lmin"),
    ]

    for sensor, raw_col, flow_col, out_col in sensor_specs:
        print(f"\nTraining model for {sensor} sensor")

        train_df = build_sensor_training_data(
            op_df, cal_df, sensor, raw_col, flow_col
        )

        model = train_hybrid_sensor_model(
            train_df, raw_col, flow_col
        )

        # Apply model to operational data
        op_df[out_col] = model.predict(
            op_df[[raw_col, flow_col]]
        )

    return op_df


plc_df = apply_all_sensor_models(plc_df, calib_df)
"""

'\ndef apply_all_sensor_models(op_df, cal_df):\n\n    sensor_specs = [\n        ("LowFlow", "LowFlowRAW", "LowFlow", "LowFlow_Lmin"),\n        ("ArgonFlow", "ArgonFlowRAW", "ArgonFlow", "ArgonFlow_Lmin"),\n        ("HighFlow", "HighFlowRAW", "HighFlow", "HighFlow_Lmin"),\n    ]\n\n    for sensor, raw_col, flow_col, out_col in sensor_specs:\n        print(f"\nTraining model for {sensor} sensor")\n\n        train_df = build_sensor_training_data(\n            op_df, cal_df, sensor, raw_col, flow_col\n        )\n\n        model = train_hybrid_sensor_model(\n            train_df, raw_col, flow_col\n        )\n\n        # Apply model to operational data\n        op_df[out_col] = model.predict(\n            op_df[[raw_col, flow_col]]\n        )\n\n    return op_df\n\n\nplc_df = apply_all_sensor_models(plc_df, calib_df)\n'

In [10]:
plc_df.to_csv(OUTPUT_CSV, index=False)

print("Corrected CSV saved to:")
print(OUTPUT_CSV)


Corrected CSV saved to:
/notebooks/Thesis-DataLogging/plc_operational_data_with_corrected_flows.csv
